# Global Solution — MLP para roubo e furto em São Paulo
Fonte: https://www.ssp.sp.gov.br/estatistica/consultas — Celulares subtraídos (2025). O alvo é **tipo de ocorrência**, não previsão de crime.

**Entradas:** bairro, período e dia da semana; excluímos rubrica e dados pessoais para evitar vazamento. O arquivo original deve ser colocado em `data/`. A seleção geográfica usa cidade do fato.

In [ ]:
from modelo import prepare, train, FEATURES, CONFIGS, OUT
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

## 1. Exploração e preparação
O código normaliza nomes de colunas, remove rubricas ambíguas, deduplica BOs, descarta datas/horas inválidas e limita a amostra a 20 mil observações quando necessário.

In [ ]:
df, arquivos = prepare()
print("Arquivos:", arquivos, "| Registros:", len(df))
display(df.head())
display(df.alvo.map({0:"Furto",1:"Roubo"}).value_counts().rename("Quantidade"))
display(df[FEATURES].isna().sum().rename("Ausentes"))

## 2. Treino, validação e comparação
Divisão estratificada 70/15/15; codificação one-hot ajustada somente no treino. Arquiteturas: [32], [64,32] e [128,64,32]. ReLU, dropout 0,2, Adam, BCEWithLogitsLoss, batch 128, máximo 60 épocas e early stopping. Escolha por F1 na validação.

In [ ]:
resultados = train()
display(pd.DataFrame(resultados["arquiteturas"]).T[["acuracia","precision","recall","f1","epocas"]])
print("Selecionada:", resultados["selecionada"])

## 3. Avaliação final e diagnóstico
A classe positiva é roubo. As métricas de teste só são calculadas após escolher a arquitetura na validação. Compare as curvas para identificar divergência entre treino e validação (overfitting) ou desempenho baixo em ambos (underfitting).

In [ ]:
print("Métricas finais no teste:", resultados["teste"])
print("Matriz de confusão:", resultados["matriz_confusao"])
display(Image(filename=str(OUT/"curvas.png")))
display(Image(filename=str(OUT/"matriz_confusao.png")))

## 4. Conclusão

O projeto utilizou 20.000 registros de ocorrências de roubo e furto de celulares na cidade de São Paulo, obtidos da base pública da SSP-SP.

Foram treinadas e comparadas três arquiteturas de redes neurais MLP. A arquitetura MLP_1, composta por uma camada oculta de 32 neurônios, apresentou o melhor desempenho na etapa de validação, considerando o F1-score.

No conjunto de teste, o modelo selecionado apresentou:

- Acurácia: 71,17%
- Precisão: 73,22%
- Recall: 89,92%
- F1-score: 80,71%

A matriz de confusão mostrou que o modelo identificou corretamente 1.810 ocorrências de roubo e 325 ocorrências de furto. Entretanto, classificou incorretamente 662 furtos como roubos e 203 roubos como furtos.

Os resultados indicam que a rede neural conseguiu identificar padrões nos dados históricos, mas ainda apresenta limitações, especialmente na diferenciação entre as duas classes.

O modelo não prevê a ocorrência futura de crimes. Sua finalidade é classificar ocorrências registradas ou hipotéticas a partir das características fornecidas.

Entre as principais limitações estão a subnotificação, os possíveis vieses dos registros policiais e a quantidade reduzida de atributos utilizados.

Ferramentas de inteligência artificial generativa foram utilizadas como apoio à programação e à documentação do projeto.